# GPT-2 LoRA Picard r5 one-click audit

This notebook starts from a blank Colab runtime, clones `papasop/fibre-nav`, checks out the frozen `v1.6.0` evidence tag, verifies the exact commit, runs the repository verifier, installs the frozen dependencies, executes the r5 ten-step audit, and downloads a result ZIP.

The r5 run is a same-cohort measurement-resolution audit of the r4 five-new-seed confirmation. It is not an independent new-seed confirmation or a universal optimizer comparison.

In [ ]:
REPO_URL = "https://github.com/papasop/fibre-nav.git"
REPO_DIR = "fibre-nav"
TAG = "v1.6.0"
EXPECTED_COMMIT = "bad9b71d6dbbc36775eb14400e14af719b58e4c5"
OUTDIR = "picard_gpt2_lora_r5_colab_results"


In [ ]:
import os
import shutil
import subprocess

shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "checkout", "--quiet", TAG], check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(head)
if head != EXPECTED_COMMIT:
    raise SystemExit(f"unexpected checkout: {head}")


In [ ]:
!python verify_picard_gpt2_lora_v1_6.py


In [ ]:
!python -m pip install -q -r evidence/audits/picard_gpt2_lora_r5_ten_step/code/requirements.txt


In [ ]:
!python external_tests/picard_gpt2_lora/COLAB_ONE_CLICK.py --source-root . --run --outdir {OUTDIR} --device cuda


In [ ]:
!python - <<'PY'
import hashlib
import json
import pathlib
import zipfile

outdir = pathlib.Path('picard_gpt2_lora_r5_colab_results')
summary = outdir / 'run_summary.json'
if not summary.is_file():
    raise SystemExit('missing run_summary.json')
data = json.loads(summary.read_text())
if data.get('scientific_status') != 'GPT2_LORA_PICARD_V0_2_6_R5_TEN_STEP_RESOLUTION_SUPPORTED':
    raise SystemExit('r5 audit did not pass the frozen supported status')

zip_path = pathlib.Path('picard_gpt2_lora_r5_colab_results.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(outdir.rglob('*')):
        if path.is_file():
            zf.write(path, path.relative_to(outdir.parent))

digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
print(json.dumps({
    'result_zip': str(zip_path),
    'sha256': digest,
    'r5_speedup': data['median_time_to_equal_loss_speedup_fraction'],
    'positive_seeds': data['positive_seed_count'],
    'fixed_budget_diagnostic': data['median_fixed_budget_speedup_fraction_diagnostic'],
}, indent=2))
PY


In [ ]:
from google.colab import files
files.download('picard_gpt2_lora_r5_colab_results.zip')
